# ARC-AGI-2 Adaptive TTT Solver
Built from scratch for ARC Prize 2026 – ARC-AGI-2.

* **Model**: public NVARC Qwen3-4B grid model (`sorokin/qwen3_4b_grids15_sft139`), LoRA r=256 test-time training per task.
* **Scheduler**: longest-first queue over 4×L4, per-task budget = remaining GPU time × task-cost share,
  online-calibrated throughput model, hard deadline + watchdog → every hidden task gets processed.
* **Decoding**: batched DFS keeping every completion with p ≥ 0.2 over D8 × colour-permutation augmentations
  (breadth-first over test outputs; extra passes when time allows), deterministic (crc32-seeded) re-scoring under 8
  augmentations, rank = votes − mean augmented NLL.
* **CPU in parallel**: verified program search (only programs exact on every train pair), merged conservatively.
* A valid `submission.json` exists from the first second and is refreshed as tasks finish.

In [ ]:
import os, sys, time, subprocess
START = time.time()
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
# transformers must not import TensorFlow in this image
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "tensorflow"], capture_output=True)
print("rerun:", RERUN, "| python", sys.version.split()[0])
!nvidia-smi -L

In [ ]:
%%writefile arcio.py
"""ARC data handling: token encoding, augmentation, submission building.

Grids are encoded directly to token ids (no string tokenization on the hot path).
The prompt layout follows the chat format the Qwen3-4B grid model was trained on:
    <|im_start|>user\n{grid}<|im_end|><|im_start|>assistant\n{grid}<|im_end|>
"""
import glob
import json
import os
import zlib

import numpy as np

MAX_SIDE = 30


# ----------------------------------------------------------------------------- paths
def find_file(name, roots=("/kaggle/input",)):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
        if hits:
            return hits[0]
    return None


def stable_seed(*parts):
    return zlib.crc32("|".join(map(str, parts)).encode()) & 0x7FFFFFFF


# ----------------------------------------------------------------------------- grids
def as_grid(a):
    return np.asarray(a, dtype=np.int64)


def is_valid_grid(a):
    return (isinstance(a, np.ndarray) and a.ndim == 2 and 1 <= a.shape[0] <= MAX_SIDE
            and 1 <= a.shape[1] <= MAX_SIDE and a.min() >= 0 and a.max() <= 9)


def grid_key(a):
    a = np.asarray(a)
    return (a.shape, a.astype(np.int8).tobytes())


# D8 symmetry group: index t -> (transpose first?, number of 90-degree rotations)
D8 = [(False, 0), (False, 1), (False, 2), (False, 3), (True, 0), (True, 1), (True, 2), (True, 3)]


def d8_fwd(a, t):
    tr, k = D8[t]
    a = np.asarray(a)
    if tr:
        a = a.T
    return np.rot90(a, k)


def d8_inv(a, t):
    tr, k = D8[t]
    a = np.rot90(np.asarray(a), -k)
    if tr:
        a = a.T
    return a


def d8_swaps_axes(t):
    tr, k = D8[t]
    return tr != (k % 2 == 1)


class Aug:
    """One augmentation: D8 transform + colour permutation + train-example order."""

    __slots__ = ("t", "perm", "inv_perm", "order")

    def __init__(self, t, perm, order):
        self.t = int(t)
        self.perm = np.asarray(perm, dtype=np.int64)
        self.inv_perm = np.argsort(self.perm)
        self.order = list(order)

    def fwd(self, g):
        return self.perm[d8_fwd(as_grid(g), self.t)]

    def inv(self, g):
        return d8_inv(self.inv_perm[as_grid(g)], self.t)

    def __repr__(self):
        return f"Aug(t={self.t}, perm={''.join(map(str, self.perm))}, order={self.order})"


def make_augs(n_train, n_colour, rng, d8=range(8), identity_first=False):
    """len(d8) * n_colour augmentations; colour permutations touch all 10 colours."""
    augs = []
    for c in range(n_colour):
        for t in d8:
            if identity_first and c == 0:
                perm, order = np.arange(10), list(range(n_train))
            else:
                perm, order = rng.permutation(10), rng.permutation(n_train).tolist()
            augs.append(Aug(t, perm, order))
    return augs


# ----------------------------------------------------------------------------- tokens
class TokenMap:
    """Token ids for digits, newline and the chat-format pieces."""

    def __init__(self, digit, nl, user_prefix, mid, end, pad):
        self.digit = np.asarray(digit, dtype=np.int64)
        self.nl = int(nl)
        self.user_prefix = list(user_prefix)   # "<|im_start|>user\n"
        self.mid = list(mid)                   # "<|im_end|><|im_start|>assistant\n"
        self.end = list(end)                   # "<|im_end|>"
        self.pad = int(pad)
        self.eos = self.end[-1]
        self.decode_vocab = {int(d): i for i, d in enumerate(self.digit)}

    @classmethod
    def from_tokenizer(cls, tok):
        enc = lambda s: tok.encode(s, add_special_tokens=False)
        digit = [enc(str(i)) for i in range(10)]
        nl = enc("\n")
        assert all(len(d) == 1 for d in digit) and len(nl) == 1, (digit, nl)
        end = enc("<|im_end|>")
        pad = tok.pad_token_id if tok.pad_token_id is not None else end[-1]
        tm = cls([d[0] for d in digit], nl[0], enc("<|im_start|>user\n"),
                 enc("<|im_end|><|im_start|>assistant\n"), end, pad)
        assert len(tm.end) == 1, tm.end
        return tm

    def grid(self, g):
        g = as_grid(g)
        h, w = g.shape
        body = np.concatenate([self.digit[g], np.full((h, 1), self.nl)], axis=1).reshape(-1)
        return body[:-1].tolist()      # no trailing newline

    def reply(self, g):
        return self.grid(g) + self.end

    def max_reply_len(self):
        return MAX_SIDE * (MAX_SIDE + 1) - 1 + len(self.end)

    def decode_grid(self, ids):
        """Token ids (without the final eos) -> grid, or None if malformed."""
        rows, row = [], []
        for t in ids:
            t = int(t)
            if t == self.nl:
                rows.append(row)
                row = []
            elif t in self.decode_vocab:
                row.append(self.decode_vocab[t])
            else:
                return None
        rows.append(row)
        if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
            return None
        a = np.asarray(rows, dtype=np.int64)
        return a if is_valid_grid(a) else None


def text_format(pairs, query=None):
    """Reference string format (used only to verify TokenMap against the tokenizer)."""
    gs = lambda g: "\n".join("".join(str(int(v)) for v in row) for row in g)
    s = "".join(f"<|im_start|>user\n{gs(p['input'])}<|im_end|><|im_start|>assistant\n{gs(p['output'])}<|im_end|>"
                for p in pairs)
    if query is not None:
        s += f"<|im_start|>user\n{gs(query)}<|im_end|><|im_start|>assistant\n"
    return s


def encode_pairs(tm, pairs, with_labels=True):
    ids, labels = [], []
    for p in pairs:
        head = tm.user_prefix + tm.grid(p["input"]) + tm.mid
        out = tm.reply(p["output"])
        ids += head + out
        labels += [-100] * len(head) + out
    return (ids, labels) if with_labels else ids


def encode_query(tm, g):
    return tm.user_prefix + tm.grid(g) + tm.mid


def augmented_pairs(task, aug):
    return [{"input": aug.fwd(task["train"][i]["input"]), "output": aug.fwd(task["train"][i]["output"])}
            for i in aug.order]


def fit_pairs(tm, pairs, extra_len, max_len):
    """Drop leading demonstration pairs until the sequence fits (keep >= 1)."""
    pairs = list(pairs)
    enc = [encode_pairs(tm, [p], with_labels=False) for p in pairs]
    total = sum(map(len, enc)) + extra_len
    while total > max_len and len(pairs) > 1:
        total -= len(enc.pop(0))
        pairs.pop(0)
    return pairs, total <= max_len


def train_sample(tm, task, aug, max_len):
    pairs, ok = fit_pairs(tm, augmented_pairs(task, aug), 0, max_len)
    if not ok:
        return None
    ids, labels = encode_pairs(tm, pairs)
    return ids, labels


def decode_prompt(tm, task, test_in, aug, max_len):
    q = encode_query(tm, aug.fwd(test_in))
    pairs, ok = fit_pairs(tm, augmented_pairs(task, aug), len(q) + tm.max_reply_len(), max_len)
    if not ok:
        return None
    return encode_pairs(tm, pairs, with_labels=False) + q


def task_tokens(task):
    """Cheap token-count estimate of all demonstration pairs (for the cost model)."""
    n = 0
    for p in task["train"]:
        for g in (p["input"], p["output"]):
            n += len(g) * (len(g[0]) + 1)
        n += 8
    return n


# ----------------------------------------------------------------------------- submission
def empty_submission(tasks):
    return {k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in t["test"]] for k, t in tasks.items()}


def validate_submission(sub, tasks):
    assert set(sub) == set(tasks), "task id mismatch"
    for k, t in tasks.items():
        assert isinstance(sub[k], list) and len(sub[k]) == len(t["test"]), k
        for e in sub[k]:
            assert set(e) == {"attempt_1", "attempt_2"}, k
            for a in e.values():
                g = np.asarray(a)
                assert g.ndim == 2 and g.size > 0 and g.dtype.kind == "i", k
    return True


def score_submission(sub, tasks, solutions):
    got, n = 0.0, 0
    for k in tasks:
        if k not in solutions:
            continue
        sols = solutions[k]
        for i, s in enumerate(sols):
            e = sub[k][i]
            got += any(np.array_equal(np.asarray(e[a]), np.asarray(s)) for a in ("attempt_1", "attempt_2")) / len(sols)
        n += 1
    return got, n


def write_json_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


In [ ]:
%%writefile symbolic.py
"""CPU program search. A program is kept only if it reproduces EVERY train pair exactly.

Each candidate is a grid function f (possibly parametrised); a consistent post colour-map is
learned on top of it, so f only has to get the geometry right.
"""
import itertools
import time

import numpy as np
from scipy.ndimage import label as cc_label

from arcio import d8_fwd

ST4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])
ST8 = np.ones((3, 3), dtype=int)


def A(g):
    return np.asarray(g, dtype=np.int64)


def bg_of(a):
    v, n = np.unique(a, return_counts=True)
    return int(v[np.argmax(n)])


# ----------------------------------------------------------------------------- primitives
def objects(a, bg, diag, multicolor):
    out = []
    st = ST8 if diag else ST4
    if multicolor:
        lab, n = cc_label(a != bg, structure=st)
        for i in range(1, n + 1):
            out.append(lab == i)
    else:
        for c in np.unique(a):
            if c == bg:
                continue
            lab, n = cc_label(a == c, structure=st)
            for i in range(1, n + 1):
                out.append(lab == i)
    return out


def bbox(mask):
    ys, xs = np.where(mask)
    return ys.min(), ys.max() + 1, xs.min(), xs.max() + 1


def crop_mask(a, mask, keep_only=False, bg=0):
    y0, y1, x0, x1 = bbox(mask)
    sub = a[y0:y1, x0:x1].copy()
    if keep_only:
        sub[~mask[y0:y1, x0:x1]] = bg
    return sub


SELECTORS = {
    "largest": lambda objs, a: max(range(len(objs)), key=lambda i: (objs[i].sum(), -bbox(objs[i])[0])),
    "smallest": lambda objs, a: min(range(len(objs)), key=lambda i: (objs[i].sum(), bbox(objs[i])[0])),
    "largest_bbox": lambda objs, a: max(range(len(objs)), key=lambda i: _bbox_area(objs[i])),
    "most_colours": lambda objs, a: max(range(len(objs)), key=lambda i: len(np.unique(a[objs[i]]))),
    "unique_colour": lambda objs, a: _unique_by(objs, a, lambda m: tuple(sorted(np.unique(a[m])))),
    "unique_shape": lambda objs, a: _unique_by(objs, a, lambda m: _shape_sig(m)),
    "unique_size": lambda objs, a: _unique_by(objs, a, lambda m: int(m.sum())),
    "top_left": lambda objs, a: min(range(len(objs)), key=lambda i: bbox(objs[i])[:3:2]),
    "bottom_right": lambda objs, a: max(range(len(objs)), key=lambda i: (bbox(objs[i])[1], bbox(objs[i])[3])),
    "densest": lambda objs, a: max(range(len(objs)), key=lambda i: objs[i].sum() / _bbox_area(objs[i])),
    "sparsest": lambda objs, a: min(range(len(objs)), key=lambda i: objs[i].sum() / _bbox_area(objs[i])),
}


def _bbox_area(m):
    y0, y1, x0, x1 = bbox(m)
    return (y1 - y0) * (x1 - x0)


def _shape_sig(m):
    y0, y1, x0, x1 = bbox(m)
    return m[y0:y1, x0:x1].tobytes() + bytes([y1 - y0, x1 - x0])


def _unique_by(objs, a, key):
    keys = [key(m) for m in objs]
    uniq = [i for i, k in enumerate(keys) if keys.count(k) == 1]
    if len(uniq) != 1:
        raise ValueError("no unique object")
    return uniq[0]


def split_by_lines(a):
    """Split on full rows/cols of one colour; returns list of equal-shape parts or None."""
    H, W = a.shape
    for c in np.unique(a):
        rows = [i for i in range(H) if (a[i] == c).all()]
        cols = [j for j in range(W) if (a[:, j] == c).all()]
        if not rows and not cols:
            continue
        rs = _runs(set(rows), H)
        cs = _runs(set(cols), W)
        parts = [a[r0:r1, c0:c1] for r0, r1 in rs for c0, c1 in cs]
        if len(parts) >= 2 and len({p.shape for p in parts}) == 1:
            return parts
    return None


def _runs(banned, n):
    out, s = [], None
    for i in range(n):
        if i in banned:
            if s is not None:
                out.append((s, i))
                s = None
        elif s is None:
            s = i
    if s is not None:
        out.append((s, n))
    return out


def equal_parts(a, k, axis):
    n = a.shape[axis]
    if n % k:
        return None
    return np.split(a, k, axis=axis)


def gravity(a, d, bg):
    b = np.rot90(a, d)
    out = np.full_like(b, bg)
    for j in range(b.shape[1]):
        col = b[:, j][b[:, j] != bg]
        if len(col):
            out[b.shape[0] - len(col):, j] = col
    return np.rot90(out, -d)


def symmetrize(a, mask_colour, kinds):
    out = a.copy()
    holes = out == mask_colour
    for _ in range(2):
        for k in kinds:
            t = {"h": np.fliplr, "v": np.flipud, "t": np.transpose, "r2": lambda z: np.rot90(z, 2)}[k](out)
            if t.shape != out.shape:
                continue
            fill = holes & (t != mask_colour)
            out[fill] = t[fill]
            holes = out == mask_colour
    if holes.any():
        raise ValueError("unfilled")
    return out


# ----------------------------------------------------------------------------- candidate programs
def programs(train):
    """Yield (name, f) candidates plausible for this task (cheap pre-filtering by shapes)."""
    ins = [A(p["input"]) for p in train]
    outs = [A(p["output"]) for p in train]
    same = all(i.shape == o.shape for i, o in zip(ins, outs))

    for t in range(8):
        yield f"d8_{t}", (lambda a, t=t: d8_fwd(a, t))

    # tilings of D8 copies: output = (ry x rx) blocks
    ratios = {(o.shape[0] / i.shape[0], o.shape[1] / i.shape[1]) for i, o in zip(ins, outs)}
    if len(ratios) == 1:
        ry, rx = next(iter(ratios))
        if ry == int(ry) and rx == int(rx) and ry * rx > 1 and ry <= 5 and rx <= 5:
            ry, rx = int(ry), int(rx)
            yield f"upscale_{ry}x{rx}", (lambda a, ry=ry, rx=rx: np.kron(a, np.ones((ry, rx), dtype=np.int64)))
            yield "kron_self", (lambda a, ry=ry, rx=rx: np.kron((a != bg_of(a)).astype(np.int64), a)
                                if a.shape == (ry, rx) else None)
            # choose a D8 transform per block independently (learned from the first pair)
            i0, o0 = ins[0], outs[0]
            h, w = i0.shape
            per_block = []
            for by in range(ry):
                for bx in range(rx):
                    blk = o0[by * h:(by + 1) * h, bx * w:(bx + 1) * w]
                    opts = [t for t in range(8) if d8_fwd(i0, t).shape == blk.shape and np.array_equal(d8_fwd(i0, t), blk)]
                    per_block.append(opts)
            if all(per_block):
                for choice in itertools.islice(itertools.product(*per_block), 16):
                    def tile(a, choice=choice, ry=ry, rx=rx):
                        rows = [np.concatenate([d8_fwd(a, choice[by * rx + bx]) for bx in range(rx)], 1) for by in range(ry)]
                        return np.concatenate(rows, 0)
                    yield f"tile_{choice}", tile
        if 1 / ry == int(1 / ry) and 1 / rx == int(1 / rx) and (ry < 1 or rx < 1):
            fy, fx = int(round(1 / ry)), int(round(1 / rx))
            yield f"downscale_{fy}x{fx}", (lambda a, fy=fy, fx=fx: a[::fy, ::fx])

    if not same:
        # crops: to content bbox, to selected objects
        yield "crop_content", (lambda a: crop_mask(a, a != bg_of(a)))
        for diag in (False, True):
            for multi in (True, False):
                for sel_name, sel in SELECTORS.items():
                    for keep in (False, True):
                        def f(a, diag=diag, multi=multi, sel=sel, keep=keep):
                            bg = bg_of(a)
                            objs = objects(a, bg, diag, multi)
                            if not objs or len(objs) > 60:
                                return None
                            return crop_mask(a, objs[sel(objs, a)], keep, bg)
                        yield f"obj_{sel_name}_{diag}_{multi}_{keep}", f
        # boolean combination of parts (split by separator lines or into equal halves)
        for splitter_name, splitter in [("lines", split_by_lines)] + [
                (f"eq{k}{ax}", (lambda a, k=k, ax=ax: equal_parts(a, k, ax))) for k in (2, 3) for ax in (0, 1)]:
            for op in ("and", "or", "xor", "nor", "a_not_b", "b_not_a"):
                def f(a, splitter=splitter, op=op):
                    parts = splitter(a)
                    if parts is None or len(parts) < 2:
                        return None
                    bgs = [bg_of(p) for p in parts]
                    m = [p != 0 for p in parts] if 0 in a else [p != b for p, b in zip(parts, bgs)]
                    x, y = m[0], m[1]
                    for z in m[2:]:
                        x = x | z if op == "or" else x & z if op == "and" else x ^ z if op == "xor" else x
                    r = {"and": x & y, "or": x | y, "xor": x ^ y, "nor": ~(x | y), "a_not_b": x & ~y,
                         "b_not_a": y & ~x}[op]
                    return r.astype(np.int64)
                yield f"bool_{splitter_name}_{op}", f
        # choose one part by property
        for splitter_name, splitter in [("lines", split_by_lines)]:
            for sel_name, key in [("most_fg", lambda p: (p != bg_of(p)).sum()), ("least_fg", lambda p: -(p != bg_of(p)).sum()),
                                  ("most_colours", lambda p: len(np.unique(p))), ("least_colours", lambda p: -len(np.unique(p)))]:
                def f(a, splitter=splitter, key=key):
                    parts = splitter(a)
                    if parts is None:
                        return None
                    ks = [key(p) for p in parts]
                    if ks.count(max(ks)) != 1:
                        return None
                    return parts[int(np.argmax(ks))]
                yield f"part_{splitter_name}_{sel_name}", f
            def f(a, splitter=splitter):
                parts = splitter(a)
                if parts is None:
                    return None
                sigs = [p.tobytes() for p in parts]
                odd = [i for i, s in enumerate(sigs) if sigs.count(s) == 1]
                return parts[odd[0]] if len(odd) == 1 else None
            yield f"part_{splitter_name}_odd", f
    else:
        for d in range(4):
            yield f"gravity_{d}", (lambda a, d=d: gravity(a, d, bg_of(a)))
        cols = set(np.unique(np.concatenate([i.ravel() for i in ins]))) - set(np.unique(np.concatenate([o.ravel() for o in outs])))
        for mc in cols:
            for kinds in (("h",), ("v",), ("h", "v"), ("t",), ("h", "v", "t"), ("r2",), ("h", "v", "t", "r2")):
                yield f"sym_{mc}_{kinds}", (lambda a, mc=mc, kinds=kinds: symmetrize(a, mc, kinds))


def learn_colour_map(preds, outs):
    cm = {}
    for p, o in zip(preds, outs):
        if p is None or p.shape != o.shape:
            return None
        for a, b in zip(p.ravel().tolist(), o.ravel().tolist()):
            if cm.setdefault(a, b) != b:
                return None
    return cm


def apply_cm(g, cm):
    lut = np.arange(max(10, g.max() + 1))
    for a, b in cm.items():
        if a < len(lut):
            lut[a] = b
    return lut[g]


def solve(task, time_limit=20.0, max_preds=2):
    """Returns per test input a list of up to `max_preds` predicted grids (as lists)."""
    t0 = time.time()
    train = task["train"]
    ins = [A(p["input"]) for p in train]
    outs = [A(p["output"]) for p in train]
    tests = [A(t["input"]) for t in task["test"]]
    found = [[] for _ in tests]
    if all(np.array_equal(o, outs[0]) for o in outs[1:]):
        return [[outs[0].tolist()] for _ in tests], ["constant"]
    names = []
    for name, f in programs(train):
        if time.time() - t0 > time_limit:
            break
        try:
            preds = [f(i) for i in ins]
            if any(p is None for p in preds):
                continue
            preds = [A(p) for p in preds]
            cm = learn_colour_map(preds, outs)
            if cm is None:
                continue
            if all(np.array_equal(i, o) for i, o in zip(ins, outs)):
                continue
            res = []
            for t in tests:
                p = f(t)
                if p is None:
                    raise ValueError
                p = apply_cm(A(p), cm)
                if p.ndim != 2 or not (1 <= p.shape[0] <= 30 and 1 <= p.shape[1] <= 30) or p.max() > 9:
                    raise ValueError
                res.append(p)
        except Exception:
            continue
        added = False
        for ti, p in enumerate(res):
            if not any(np.array_equal(p, q) for q in found[ti]) and len(found[ti]) < max_preds:
                found[ti].append(p)
                added = True
        if added:
            names.append(name)
        if all(len(f_) >= max_preds for f_ in found):
            break
    return [[g.tolist() for g in f_] for f_ in found], names


def solve_many(tasks, time_limit=20.0):
    out = {}
    for k, t in tasks.items():
        try:
            preds, names = solve(t, time_limit)
            if any(preds):
                out[k] = {"preds": preds, "programs": names}
        except Exception:
            pass
    return out


In [ ]:
%%writefile engine.py
"""GPU worker: per-task test-time training + DFS decoding + augmentation re-scoring.

Run as:  python engine.py --rank R --run-dir DIR
Coordinates with the other workers only through files in DIR (claims, status, results).
"""
import argparse
import json
import math
import os
import pickle
import sys
import time
import traceback

import numpy as np

from arcio import (Aug, TokenMap, as_grid, d8_swaps_axes, decode_prompt, encode_query, grid_key,
                   make_augs, stable_seed, task_tokens, text_format, encode_pairs, train_sample)

MAX_SEQ = 8192
LORA = dict(r=256, lora_alpha=32, lora_dropout=0.0, bias="none", use_rslora=True,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
                            "embed_tokens", "lm_head"])
LR = 5e-5
WARMUP = 0.1
MAX_NLL = -math.log(0.2)       # DFS keeps continuations with cumulative prob >= 0.2
DFS_CAP = 540.0                # seconds per DFS call
LONG_PROMPT = 4500             # above this many tokens, batch 2 sequences instead of 4 (L4 memory)
N_SCORE_AUG = 8


def log(rank, msg):
    print(f"[gpu{rank} {time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ============================================================================ backends
class Backend:
    """Wraps model loading and the train/inference mode switches."""

    def __init__(self, kind, model_path, device, seed=42):
        import torch
        self.torch = torch
        self.kind = kind
        self.device = device
        torch.manual_seed(seed)
        if kind == "unsloth":
            from unsloth import FastLanguageModel
            self.FLM = FastLanguageModel
            model, tok = FastLanguageModel.from_pretrained(
                model_name=model_path, full_finetuning=False, load_in_4bit=False, local_files_only=True,
                use_gradient_checkpointing=False, max_seq_length=MAX_SEQ)
            model = FastLanguageModel.get_peft_model(
                model, use_gradient_checkpointing=False, random_state=seed, loftq_config=None, **LORA)
        else:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            from peft import LoraConfig, get_peft_model
            tok = AutoTokenizer.from_pretrained(model_path)
            model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float32).to(device)
            cfg = dict(LORA)
            cfg["target_modules"] = [m for m in cfg["target_modules"] if m.endswith("proj")]
            model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", modules_to_save=["embed_tokens", "lm_head"], **cfg))
        if kind == "unsloth":
            for p in model.parameters():
                if p.dtype == torch.float32:
                    p.data = p.data.to(torch.bfloat16)
        from peft import get_peft_model_state_dict
        self.init_state = {k: v.detach().clone() for k, v in get_peft_model_state_dict(model).items()}
        self.model, self.tok = model, tok

    def reset(self):
        from peft import set_peft_model_state_dict
        # peft mutates (renames/deletes keys of) the dict it is given -> always pass a fresh shallow copy;
        # load_state_dict copies the tensor values, so the stored tensors are never modified
        set_peft_model_state_dict(self.model, dict(self.init_state))

    def train_mode(self):
        if self.kind == "unsloth":
            self.model = self.FLM.for_training(self.model)
        self.model.train()

    def infer_mode(self):
        if self.kind == "unsloth":
            self.model = self.FLM.for_inference(self.model)
        self.model.eval()


# ============================================================================ training
def lr_at(step, total):
    warm = int(math.ceil(WARMUP * total))
    if step < warm:
        return LR * step / max(1, warm)
    prog = (step - warm) / max(1, total - warm)
    return LR * 0.5 * (1.0 + math.cos(math.pi * prog))


def ttt(be, samples, deadline):
    """One epoch of LoRA fine-tuning over `samples` (batch size 1), cosine schedule."""
    torch = be.torch
    be.reset()
    be.train_mode()
    params = [p for p in be.model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)
    total, done, losses = len(samples), 0, []
    for step, (ids, labels) in enumerate(samples):
        if time.time() > deadline:
            break
        for g in opt.param_groups:
            g["lr"] = lr_at(step, total)
        x = torch.tensor([ids], device=be.device)
        y = torch.tensor([labels], device=be.device)
        out = be.model(input_ids=x, attention_mask=torch.ones_like(x), labels=y)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach()))
        done += 1
    del opt
    be.infer_mode()
    if be.device != "cpu":
        torch.cuda.empty_cache()
    return done, (float(np.mean(losses[-16:])) if losses else float("nan"))


# ============================================================================ decoding
class Searcher:
    """Batched depth-first search over grid tokens, keeping every completion with p >= 0.2."""

    def __init__(self, be, tm, deadline):
        self.be, self.tm = be, tm
        self.allowed = [int(d) for d in tm.digit] + [tm.nl, tm.eos]
        self.allowed_t = be.torch.tensor(self.allowed, device=be.device)
        self.deadline = deadline
        self.calls = 0

    def _lp(self, logits):
        torch = self.be.torch
        lf = logits.float()
        return (lf.index_select(-1, self.allowed_t) - torch.logsumexp(lf, -1, keepdim=True)).cpu().numpy()

    def run(self, prompts, max_new):
        torch = self.be.torch
        with torch.inference_mode():
            x = torch.tensor(prompts, device=self.be.device)
            out = self.be.model(input_ids=x, use_cache=True, return_dict=True)
            lp = self._lp(out.logits[:, -1])
            found = self._search(lp, [0.0] * len(prompts), x.shape[1], out.past_key_values, max_new)
        return [sorted(f, key=lambda z: z[0]) for f in found]

    def _search(self, lp, base, pos, cache, left):
        torch, tm = self.be.torch, self.tm
        n = lp.shape[0]
        found = [[] for _ in range(n)]
        queues = []
        for i in range(n):
            q = []
            if base[i] is not None:
                for j, t in enumerate(self.allowed):
                    s = base[i] - float(lp[i, j])
                    if s < MAX_NLL:
                        if t == tm.eos:
                            found[i].append((s, [t]))
                        elif left > 1:
                            q.append((s, t))
            q.sort()
            queues.append(q)
        while any(queues) and time.time() < self.deadline:
            toks, scores = [], []
            for q in queues:
                if q:
                    s, t = q.pop(0)
                    toks.append(t)
                    scores.append(s)
                else:
                    toks.append(tm.pad)
                    scores.append(None)
            if hasattr(cache, "crop"):          # HF DynamicCache is mutated in place
                cache.crop(pos)
            out = self.be.model(input_ids=torch.tensor(toks, device=self.be.device).view(n, 1),
                                position_ids=torch.full((n, 1), pos, device=self.be.device),
                                past_key_values=cache, use_cache=True, return_dict=True)
            self.calls += 1
            sub = self._search(self._lp(out.logits[:, -1]), scores, pos + 1, out.past_key_values, left - 1)
            for i in range(n):
                for s, suf in sub[i]:
                    found[i].append((s, [toks[i]] + suf))
        return found


def seq_nll(be, tm, pairs_list, bs=4):
    """Teacher-forced NLL of the reply part. pairs_list: [(prompt_ids, reply_ids)]."""
    torch = be.torch
    res = []
    with torch.inference_mode():
        for b in range(0, len(pairs_list), bs):
            chunk = pairs_list[b:b + bs]
            seqs = [p + r for p, r in chunk]
            L = max(map(len, seqs))
            x = torch.tensor([s + [tm.pad] * (L - len(s)) for s in seqs], device=be.device)
            logits = be.model(input_ids=x, use_cache=False, return_dict=True).logits.float()
            norm = torch.logsumexp(logits, -1)
            for i, (p, r) in enumerate(chunk):
                pos = torch.arange(len(p) - 1, len(p) - 1 + len(r), device=be.device)
                tgt = torch.tensor(r, device=be.device)
                res.append(float(-(logits[i, pos, tgt] - norm[i, pos]).sum()))
    return res


# ============================================================================ planning
class Rates:
    """Online-calibrated throughput model (seconds)."""

    def __init__(self):
        self.train = 0.10 / 128      # s per sample-token
        self.dec_fixed = 3.0         # s per augmentation per output
        self.dec_tok = 0.03          # s per augmentation per output-token

    def upd_train(self, secs, tokens):
        if tokens > 0 and secs > 1:
            self.train = 0.6 * self.train + 0.4 * secs / tokens

    def upd_dec(self, secs, n_aug, out_tok):
        pred = n_aug * (self.dec_fixed + self.dec_tok * out_tok)
        if pred > 0 and secs > 1:
            f = min(3.0, max(0.33, secs / pred)) ** 0.4
            self.dec_fixed *= f
            self.dec_tok *= f


def est_out_tokens(task, test_in):
    tr = task["train"]
    tok = lambda g: len(g) * (len(g[0]) + 1)
    if all(np.shape(p["input"]) == np.shape(p["output"]) for p in tr):
        return tok(test_in)
    return int(np.mean([tok(p["output"]) for p in tr]))


def task_cost(task, rates, n_train=128, n_dec=16):
    T = min(task_tokens(task), MAX_SEQ)
    dec = sum(n_dec * (rates.dec_fixed + rates.dec_tok * est_out_tokens(task, t["input"])) for t in task["test"])
    return rates.train * T * n_train + dec + 25.0


def plan(task, rates, budget):
    """Choose (n_train, n_dec) that fits the budget; extra decode passes use leftover time."""
    for n_train, n_dec in [(128, 16), (96, 16), (64, 16), (64, 8), (48, 8), (32, 8), (16, 8)]:
        if task_cost(task, rates, n_train, n_dec) <= budget:
            return n_train, n_dec
    return 16, 8


# ============================================================================ per-task solve
def solve_task(be, tm, key, task, budget, rates, rank, safe=False):
    t0 = time.time()
    deadline = t0 + budget
    n_train, n_dec = (32, 8) if safe else plan(task, rates, budget)
    if be.device != "cpu":
        be.torch.cuda.reset_peak_memory_stats()
    rng = np.random.default_rng(stable_seed(key, "train"))
    n_pairs = len(task["train"])

    # ---- test-time training
    augs = make_augs(n_pairs, (n_train + 7) // 8, rng)
    rng.shuffle(augs)
    samples = [s for s in (train_sample(tm, task, a, MAX_SEQ) for a in augs[:n_train]) if s is not None]
    est_train = rates.train * sum(len(s[0]) for s in samples)
    train_deadline = t0 + min(0.9 * budget, max(0.65 * budget, 1.3 * est_train))
    ts = time.time()
    steps, loss = ttt(be, samples, train_deadline)
    t_train = time.time() - ts
    rates.upd_train(t_train, sum(len(s[0]) for s in samples[:steps]))

    # ---- decoding: passes of 8 D8 augmentations with a fresh colour permutation each
    max_new = tm.max_reply_len() + 1
    outputs = []
    for ti, test in enumerate(task["test"]):
        outputs.append({"cands": {}, "n_aug": 0, "est_tok": est_out_tokens(task, test["input"])})
    drng = np.random.default_rng(stable_seed(key, "decode"))
    max_passes = 4
    passes = [make_augs(n_pairs, 1, drng) for _ in range(max_passes)]
    batches = []  # (pass, test index, [augs])
    for p in range(max_passes):          # breadth first: every output gets 4 augs before any gets 8
        for swap in (False, True):
            for ti in range(len(task["test"])):
                batches.append((p, ti, [a for a in passes[p] if d8_swaps_axes(a.t) == swap]))
    min_passes = n_dec // 8
    ts = time.time()
    for p, ti, group in batches:
        now = time.time()
        if p >= min_passes:
            # extra passes only while the remaining budget comfortably covers them
            est = len(group) * (rates.dec_fixed + rates.dec_tok * outputs[ti]["est_tok"]) * 1.3
            if now + est > deadline:
                continue
        elif now > deadline + 60:
            break
        test_in = task["test"][ti]["input"]
        prompts = [(a, decode_prompt(tm, task, test_in, a, MAX_SEQ - max_new)) for a in group]
        prompts = [(a, pr) for a, pr in prompts if pr is not None]
        by_len = {}
        for a, pr in prompts:
            by_len.setdefault(len(pr), []).append((a, pr))
        chunks = []
        for L, items in sorted(by_len.items()):
            bs = 4 if L <= LONG_PROMPT else 2
            chunks += [items[i:i + bs] for i in range(0, len(items), bs)]
        for items in chunks:
            search = Searcher(be, tm, min(time.time() + DFS_CAP, max(deadline, time.time() + 90)))
            found = search.run([pr for _, pr in items], max_new)
            outputs[ti]["n_aug"] += len(items)
            for (a, _), beams in zip(items, found):
                for nll, toks in beams:
                    g = tm.decode_grid(toks[:-1])
                    if g is None:
                        continue
                    g = a.inv(g)
                    k = grid_key(g)
                    c = outputs[ti]["cands"].setdefault(k, {"grid": g, "count": 0, "beam": [], "aug_nll": None})
                    c["count"] += 1
                    c["beam"].append(nll)
    t_dec = time.time() - ts

    # ---- re-score each unique candidate under fixed augmentations
    ts = time.time()
    for ti, test in enumerate(task["test"]):
        srng = np.random.default_rng(stable_seed(key, ti, "score"))
        saugs = make_augs(n_pairs, 1, srng)
        for c in sorted(outputs[ti]["cands"].values(), key=lambda c: -c["count"]):
            if time.time() > deadline + 120:
                break
            items = []
            for a in saugs:
                pr = decode_prompt(tm, task, test["input"], a, MAX_SEQ - max_new)
                if pr is not None:
                    items.append((pr, tm.reply(a.fwd(c["grid"]))))
            if items:
                long_ = max(len(p) + len(r) for p, r in items) > LONG_PROMPT
                c["aug_nll"] = float(np.mean(seq_nll(be, tm, items, bs=2 if long_ else 4)))
    t_score = time.time() - ts
    tok_sum = sum(o["n_aug"] * o["est_tok"] for o in outputs)
    rates.upd_dec(t_dec + t_score, max(1, sum(o["n_aug"] for o in outputs)),
                  tok_sum / max(1, sum(o["n_aug"] for o in outputs)))

    result = []
    for o in outputs:
        ranked = rank_candidates(list(o["cands"].values()))
        result.append([{"grid": c["grid"].tolist(), "count": c["count"], "aug_nll": c["aug_nll"],
                        "beam": float(min(c["beam"])), "score": c["score"]} for c in ranked[:8]])
    info = dict(budget=round(budget), n_train=n_train, steps=steps, loss=round(loss, 5), t_train=round(t_train),
                t_dec=round(t_dec), t_score=round(t_score), total=round(time.time() - t0),
                n_aug=[o["n_aug"] for o in outputs], n_cand=[len(o["cands"]) for o in outputs],
                mem_gb=round(be.torch.cuda.max_memory_allocated() / 2**30, 1) if be.device != "cpu" else 0)
    return result, info


def rank_candidates(cands):
    for c in cands:
        nll = c["aug_nll"] if c["aug_nll"] is not None else 50.0
        c["score"] = c["count"] - nll
    return sorted(cands, key=lambda c: -c["score"])


# ============================================================================ scheduling (file based)
class Board:
    def __init__(self, run_dir, rank):
        self.dir, self.rank = run_dir, rank
        for sub in ("claims", "status", "results"):
            os.makedirs(os.path.join(run_dir, sub), exist_ok=True)
        with open(os.path.join(run_dir, "schedule.json")) as f:
            self.sched = json.load(f)

    def attempts(self, key):
        try:
            with open(os.path.join(self.dir, "attempts", key)) as f:
                return int(f.read() or 0)
        except Exception:
            return 0

    def release(self, key):
        """Put a failed task back in the queue (at most one retry)."""
        os.makedirs(os.path.join(self.dir, "attempts"), exist_ok=True)
        n = self.attempts(key) + 1
        with open(os.path.join(self.dir, "attempts", key), "w") as f:
            f.write(str(n))
        if n < 2:
            try:
                os.remove(os.path.join(self.dir, "claims", key))
            except FileNotFoundError:
                pass

    def claim_next(self):
        for key in self.sched["order"]:
            try:
                fd = os.open(os.path.join(self.dir, "claims", key), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                os.write(fd, str(self.rank).encode())
                os.close(fd)
                return key
            except FileExistsError:
                continue
        return None

    def unclaimed_cost(self):
        claimed = set(os.listdir(os.path.join(self.dir, "claims")))
        return sum(c for k, c in self.sched["cost"].items() if k not in claimed)

    def busy_until(self):
        out = {}
        for f in os.listdir(os.path.join(self.dir, "status")):
            try:
                with open(os.path.join(self.dir, "status", f)) as fh:
                    out[f] = json.load(fh).get("busy_until", 0)
            except Exception:
                pass
        return out

    def set_status(self, **kw):
        path = os.path.join(self.dir, "status", f"gpu{self.rank}")
        with open(path + ".tmp", "w") as f:
            json.dump(kw, f)
        os.replace(path + ".tmp", path)

    def budget(self, key, t_end, n_workers):
        now = time.time()
        busy = self.busy_until()
        others = [max(0.0, t_end - max(now, b)) for w, b in busy.items() if w != f"gpu{self.rank}"]
        others += [t_end - now] * max(0, n_workers - 1 - len(others))
        gpu_left = (t_end - now) + sum(others)
        share = self.sched["cost"][key] / max(1e-9, self.unclaimed_cost() + self.sched["cost"][key])
        b = 0.95 * gpu_left * share
        return max(30.0, min(b, t_end - now - 20, 2.2 * self.sched["cost"][key], 2400))


def cleanup(be):
    import gc
    gc.collect()
    if be.device != "cpu":
        be.torch.cuda.empty_cache()
    try:
        be.infer_mode()
    except Exception:
        pass


# ============================================================================ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rank", type=int, required=True)
    ap.add_argument("--run-dir", required=True)
    args = ap.parse_args()
    with open(os.path.join(args.run_dir, "config.json")) as f:
        cfg = json.load(f)
    rank = args.rank
    board = Board(args.run_dir, rank)
    t_end = cfg["t_end"]

    # serialize backend import/initialisation across workers (unsloth patches files on import)
    if rank > 0:
        prev = os.path.join(args.run_dir, "status", f"ready{rank - 1}")
        t_wait = time.time()
        while not os.path.exists(prev) and time.time() < min(t_end, t_wait + 900):
            time.sleep(3)
    t_load = time.time()
    device = "cuda" if cfg["backend"] == "unsloth" else "cpu"
    try:
        be = Backend(cfg["backend"], cfg["model_path"], device)
    finally:   # never block the next worker, even if this one failed to load
        open(os.path.join(args.run_dir, "status", f"ready{rank}"), "w").close()
    tm = TokenMap.from_tokenizer(be.tok)
    log(rank, f"model ready in {time.time() - t_load:.0f}s; tokens digit={tm.digit.tolist()} nl={tm.nl} "
              f"eos={tm.eos} pad={tm.pad} user={tm.user_prefix} mid={tm.mid}")

    with open(cfg["tasks_path"]) as f:
        tasks = json.load(f)
    # verify direct token encoding against the tokenizer on one real task
    k0 = sorted(tasks)[0]
    ref = be.tok.encode(text_format(tasks[k0]["train"][:1], tasks[k0]["test"][0]["input"]), add_special_tokens=False)
    mine = encode_pairs(tm, tasks[k0]["train"][:1], with_labels=False) + encode_query(tm, tasks[k0]["test"][0]["input"])
    if ref != mine:
        log(rank, f"WARNING token encoding mismatch on {k0}: {ref[:24]} vs {mine[:24]}")

    rates = Rates()
    n_done, n_fail_row = 0, 0
    while time.time() < t_end - 60:
        key = board.claim_next()
        if key is None:
            break
        budget = board.budget(key, t_end, cfg["n_workers"])
        board.set_status(busy_until=time.time() + budget, task=key)
        try:
            try:
                result, info = solve_task(be, tm, key, tasks[key], budget, rates, rank)
            except Exception:
                log(rank, f"{key} first attempt failed, retrying in safe mode\n{traceback.format_exc()[-1500:]}")
                cleanup(be)
                left = min(budget, t_end - time.time() - 30)
                if left < 60:
                    raise
                result, info = solve_task(be, tm, key, tasks[key], left, rates, rank, safe=True)
                info["safe"] = True
            with open(os.path.join(args.run_dir, "results", key + ".pkl.tmp"), "wb") as f:
                pickle.dump({"key": key, "outputs": result, "info": info}, f)
            os.replace(os.path.join(args.run_dir, "results", key + ".pkl.tmp"),
                       os.path.join(args.run_dir, "results", key + ".pkl"))
            n_done += 1
            n_fail_row = 0
            log(rank, f"{key} {info}")
        except Exception:
            log(rank, f"{key} FAILED\n{traceback.format_exc()[-1500:]}")
            board.release(key)
            board.set_status(busy_until=0, task=None)
            n_fail_row += 1
            if n_fail_row >= 3:      # something is broken in this process: exit non-zero -> main respawns a fresh one
                log(rank, "3 consecutive failures, exiting for a fresh restart")
                sys.exit(3)
            cleanup(be)
        board.set_status(busy_until=0, task=None)
    log(rank, f"finished {n_done} tasks")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile runner.py
"""Orchestrator: schedule -> 4 GPU workers + CPU symbolic search -> merged, validated submission."""
import glob
import json
import os
import pickle
import subprocess
import sys
import time
from multiprocessing import Pool

import numpy as np

from arcio import empty_submission, find_file, score_submission, validate_submission, write_json_atomic
from engine import Rates, task_cost

SRC = os.path.dirname(os.path.abspath(__file__))


def say(msg):
    print(f"[main {time.strftime('%H:%M:%S')}] {msg}", flush=True)


def locate_model():
    hits = sorted(glob.glob("/kaggle/input/**/qwen3_4b_grids15_sft139/**/config.json", recursive=True))
    if not hits:
        hits = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))
    assert hits, "model not found under /kaggle/input"
    return os.path.dirname(hits[0])


def unsloth_path(env):
    """None if a plain subprocess already finds unsloth (same env the workers get), else its directory."""
    probe = "import importlib.util,sys; s=importlib.util.find_spec('unsloth'); print(s.origin if s else ''); sys.exit(0 if s else 1)"
    r = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True, timeout=120)
    if r.returncode == 0:
        say(f"unsloth found at {r.stdout.strip()}")
        return None
    hits = sorted(glob.glob("/kaggle/input/**/unsloth/__init__.py", recursive=True))
    assert hits, "unsloth not importable and not found under /kaggle/input"
    say(f"unsloth added from {hits[0]}")
    return os.path.dirname(os.path.dirname(hits[0]))


def n_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=60).stdout
        return max(1, sum(1 for l in out.splitlines() if l.startswith("GPU")))
    except Exception:
        return 1


def _sym_one(item):
    import symbolic
    k, t = item
    try:
        preds, names = symbolic.solve(t, 20.0)
        return k, preds, names
    except Exception:
        return k, [[] for _ in t["test"]], []


def run_symbolic(tasks, procs=6):
    with Pool(procs) as p:
        res = p.map(_sym_one, sorted(tasks.items()), chunksize=2)
    return {k: {"preds": pr, "programs": nm} for k, pr, nm in res if any(pr)}


def load_results(run_dir):
    out = {}
    for f in glob.glob(os.path.join(run_dir, "results", "*.pkl")):
        try:
            with open(f, "rb") as fh:
                r = pickle.load(fh)
            out[r["key"]] = r
        except Exception:
            pass
    return out


def same(a, b):
    return np.array_equal(np.asarray(a), np.asarray(b))


def merge(tasks, results, symb):
    sub = empty_submission(tasks)
    stats = {"model": 0, "symbolic_only": 0, "symbolic_swap": 0, "fallback": 0}
    for k, t in tasks.items():
        for i, test in enumerate(t["test"]):
            model = [c["grid"] for c in results[k]["outputs"][i]] if k in results and i < len(results[k]["outputs"]) else []
            sym = symb.get(k, {}).get("preds", [[]] * len(t["test"]))[i] if k in symb else []
            att = list(model[:2])
            if att:
                stats["model"] += 1
            for s in sym[:1]:
                if not any(same(s, a) for a in att):
                    if len(att) < 2:
                        stats["symbolic_only" if not att else "symbolic_swap"] += 1
                        att.append(s)
                    else:
                        att[1] = s
                        stats["symbolic_swap"] += 1
            for s in sym[1:]:
                if len(att) < 2 and not any(same(s, a) for a in att):
                    att.append(s)
            if not att:
                stats["fallback"] += 1
                att = [test["input"]]
            if len(att) < 2:
                att.append(att[0])
            sub[k][i] = {"attempt_1": [[int(v) for v in r] for r in att[0]],
                         "attempt_2": [[int(v) for v in r] for r in att[1]]}
    return sub, stats


def run(start_time, rerun, dev_keys=None, dev_minutes=40, backend="unsloth", model_path=None, out_path="submission.json",
        run_dir="/kaggle/working/run", time_limit=12 * 3600, margin=30 * 60, kill_grace=8 * 60):
    for sub_dir in ("", "claims", "status", "results", "attempts"):
        os.makedirs(os.path.join(run_dir, sub_dir), exist_ok=True)
    test_file = find_file("arc-agi_test_challenges.json", roots=(os.environ.get("ARC_INPUT", "/kaggle/input"),))
    comp = os.path.dirname(test_file)
    with open(test_file) as f:
        test_tasks = json.load(f)
    say(f"rerun={bool(rerun)} test tasks={len(test_tasks)} data={comp}")

    # 0. a valid submission exists from the very beginning
    write_json_atomic(empty_submission(test_tasks), out_path)

    sim = os.environ.get("ARC_SIM_RERUN")       # commit-time rehearsal of the rerun path on eval tasks
    sim_solutions = None
    if sim:
        with open(os.path.join(comp, "arc-agi_evaluation_challenges.json")) as f:
            test_tasks = json.load(f)
        with open(os.path.join(comp, "arc-agi_evaluation_solutions.json")) as f:
            sim_solutions = json.load(f)
        rerun, time_limit, margin = True, float(sim) * 60, 5 * 60
        start_time = time.time()
        say(f"SIMULATED RERUN on {len(test_tasks)} eval tasks, {sim} min")
    if rerun:
        gpu_tasks, dev_solutions = test_tasks, None
        t_end = start_time + time_limit - margin
    else:
        with open(os.path.join(comp, "arc-agi_evaluation_challenges.json")) as f:
            ev = json.load(f)
        with open(os.path.join(comp, "arc-agi_evaluation_solutions.json")) as f:
            evs = json.load(f)
        gpu_tasks = {k: ev[k] for k in dev_keys}
        dev_solutions = {k: evs[k] for k in dev_keys}
        t_end = time.time() + dev_minutes * 60

    with open(os.path.join(run_dir, "tasks.json"), "w") as f:
        json.dump(gpu_tasks, f)
    rates = Rates()
    cost = {k: task_cost(t, rates) for k, t in gpu_tasks.items()}
    order = sorted(cost, key=lambda k: -cost[k])
    with open(os.path.join(run_dir, "schedule.json"), "w") as f:
        json.dump({"order": order, "cost": cost}, f)
    ng = n_gpus() if backend == "unsloth" else 2
    model_path = model_path or locate_model()
    cfg = dict(t_end=t_end, n_workers=ng, backend=backend, model_path=model_path,
               tasks_path=os.path.join(run_dir, "tasks.json"))
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f)
    say(f"gpus={ng} model={model_path} gpu tasks={len(gpu_tasks)} est. GPU-s={sum(cost.values()):.0f} "
        f"available GPU-s={(t_end - time.time()) * ng:.0f} end in {(t_end - time.time()) / 3600:.2f}h")

    # 1. launch GPU workers
    env = dict(os.environ, PYTHONHASHSEED="0", UNSLOTH_DISABLE_STATISTICS="1", OMP_NUM_THREADS="12",
               TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas")
    extra = [SRC] + ([unsloth_path(env)] if backend == "unsloth" else [])
    env["PYTHONPATH"] = os.pathsep.join([p for p in extra if p] + ([env["PYTHONPATH"]] if env.get("PYTHONPATH") else []))
    procs, logs = [], []

    def launch(r):
        e = dict(env, CUDA_VISIBLE_DEVICES=str(r)) if backend == "unsloth" else env
        lf = open(os.path.join(run_dir, f"gpu{r}.log"), "a")
        logs.append(lf)
        return subprocess.Popen([sys.executable, "-u", os.path.join(SRC, "engine.py"), "--rank", str(r),
                                 "--run-dir", run_dir], env=e, stdout=lf, stderr=subprocess.STDOUT, cwd=SRC)

    for r in range(ng):
        procs.append(launch(r))
    restarts = [0] * ng

    # 2. symbolic search on CPU while the models load
    ts = time.time()
    symb = run_symbolic(test_tasks)
    symb_dev = run_symbolic(gpu_tasks) if not rerun else symb
    say(f"symbolic: {len(symb)}/{len(test_tasks)} test tasks with a verified program ({time.time() - ts:.0f}s)")

    # 3. monitor; refresh the submission as results arrive; hard deadline
    t_kill = t_end + kill_grace
    last_n, last_print = -1, 0
    printed = {r: 0 for r in range(ng)}
    while True:
        alive = [p.poll() is None for p in procs]
        unclaimed = len(gpu_tasks) - len(os.listdir(os.path.join(run_dir, "claims")))
        for r, a in enumerate(alive):   # respawn crashed workers while work and time remain
            if not a and procs[r].returncode != 0 and restarts[r] < 3 and unclaimed > 0 and t_end - time.time() > 1200:
                restarts[r] += 1
                say(f"worker {r} exited with code {procs[r].returncode}; respawn #{restarts[r]}")
                procs[r] = launch(r)
                alive[r] = True
        for r, a in enumerate(alive):   # a dead worker contributes no GPU time to the budgets
            if not a:
                write_json_atomic({"busy_until": 1e18, "dead": True}, os.path.join(run_dir, "status", f"gpu{r}"))
        results = load_results(run_dir)
        if len(results) != last_n and rerun:
            sub, _ = merge(test_tasks, results, symb)
            write_json_atomic(sub, out_path)
            last_n = len(results)
        if time.time() - last_print > 300 or not any(alive):
            say(f"done {len(results)}/{len(gpu_tasks)} alive={sum(alive)} left={(t_end - time.time()) / 60:.0f}min")
            for r in range(ng):
                with open(os.path.join(run_dir, f"gpu{r}.log")) as fh:
                    lines = fh.read().splitlines()
                for line in lines[printed[r]:]:
                    if line.startswith("[gpu") or "Error" in line or "Traceback" in line:
                        print(line[:400], flush=True)
                printed[r] = len(lines)
            last_print = time.time()
        if not any(alive):
            break
        if time.time() > t_kill:
            say("hard deadline: terminating workers")
            for p in procs:
                if p.poll() is None:
                    p.kill()
            break
        time.sleep(20)
    for lf in logs:
        lf.close()

    # 4. final assembly
    results = load_results(run_dir)
    if rerun:
        sub, stats = merge(test_tasks, results, symb)
        if sim_solutions:
            got, n = score_submission(sub, test_tasks, sim_solutions)
            say(f"SIM SCORE (eval, inflated) {got:.2f}/{n} processed {len(results)}/{len(test_tasks)}")
            infos = [r["info"] for r in results.values()]
            say(f"SIM max mem {max([i.get('mem_gb', 0) for i in infos] or [0])} GB, safe-mode tasks "
                f"{sum(1 for i in infos if i.get('safe'))}")
    else:
        sub, stats = merge(test_tasks, {}, symb)
        dev_sub, dev_stats = merge(gpu_tasks, results, symb_dev)
        model_only, _ = merge(gpu_tasks, results, {})
        got, n = score_submission(dev_sub, gpu_tasks, dev_solutions)
        got_m, _ = score_submission(model_only, gpu_tasks, dev_solutions)
        say(f"DEV (eval subset, inflated: the model has likely seen these tasks): merged {got:.2f}/{n}  "
            f"model-only {got_m:.2f}/{n}  processed {len(results)}/{n}  stats={dev_stats}")
        for k in dev_keys:
            if k in results:
                say(f"  {k} {results[k]['info']}")
    validate_submission(sub, test_tasks)
    write_json_atomic(sub, out_path)
    say(f"submission written: {out_path} tasks={len(sub)} merge={stats} processed={len(results)}/{len(gpu_tasks)} "
        f"elapsed={(time.time() - start_time) / 3600:.2f}h")
    return sub


In [ ]:
sys.path.insert(0, "/kaggle/working")
import runner
DEV_KEYS = ['0934a4d8', '36a08778', '981571dc', 'aa4ec2a5', '135a2760', '20270e3b', '2ba387bc', '3e6067c3']   # commit run only: eval tasks used as a correctness check
sub = runner.run(START, RERUN, dev_keys=DEV_KEYS, dev_minutes=42,
                 out_path="/kaggle/working/submission.json", run_dir="/kaggle/working/run")
print("tasks in submission.json:", len(sub))